## Section 1 — Setup

In [ ]:
!pip install torch numpy mlflow scikit-learn
from google.colab import drive
drive.mount('/content/drive')
DRIVE_PATH = '/content/drive/MyDrive/anomaly_detection/'

## Section 2 — Upload Data

In [ ]:
import numpy as np
import os

print("Loading data from Drive...")
X_train = np.load(os.path.join(DRIVE_PATH, "X_train.npy"))
X_test = np.load(os.path.join(DRIVE_PATH, "X_test.npy"))
y_test = np.load(os.path.join(DRIVE_PATH, "y_test.npy"))

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

## Section 3 — Model Definition

In [ ]:
import torch
import torch.nn as nn

class LSTMAutoencoder(nn.Module):
    def __init__(self, n_features, latent_dim, n_layers):
        super(LSTMAutoencoder, self).__init__()
        self.n_features = n_features
        self.latent_dim = latent_dim
        self.n_layers = n_layers
        
        # Encoder
        self.encoder = nn.LSTM(
            input_size=n_features,
            hidden_size=latent_dim,
            num_layers=n_layers,
            batch_first=True
        )
        
        # Decoder
        self.decoder = nn.LSTM(
            input_size=latent_dim,
            hidden_size=n_features,
            num_layers=n_layers,
            batch_first=True
        )
        
    def forward(self, x):
        # Encode
        _, (hidden, _) = self.encoder(x)
        
        # Get last hidden state
        last_hidden = hidden[-1]
        
        # Repeat context vector for seq_len
        seq_len = x.shape[1]
        decoder_input = last_hidden.unsqueeze(1).repeat(1, seq_len, 1)
        
        # Decode
        reconstruction, _ = self.decoder(decoder_input)
        return reconstruction

## Section 4 — Training

In [ ]:
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

N_FEATURES = 8
LATENT_DIM = 32
N_LAYERS = 2
WINDOW_SIZE = 30
BATCH_SIZE = 64
EPOCHS = 50
LEARNING_RATE = 1e-3

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
train_dataset = TensorDataset(X_train_tensor, X_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

model = LSTMAutoencoder(N_FEATURES, LATENT_DIM, N_LAYERS).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print("Starting training...")
model.train()
epoch_losses = []

for epoch in range(1, EPOCHS + 1):
    epoch_loss = 0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
    avg_loss = epoch_loss / len(train_loader)
    epoch_losses.append(avg_loss)
    
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch [{epoch}/{EPOCHS}], Loss: {avg_loss:.6f}")

plt.figure(figsize=(10, 5))
plt.plot(range(1, EPOCHS + 1), epoch_losses, label='Training Loss')
plt.title('Autoencoder Training Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.legend()
plt.show()

## Section 5 — Threshold Computation

In [ ]:
model.eval()
with torch.no_grad():
    X_train_gpu = X_train_tensor.to(device)
    train_preds = model(X_train_gpu)
    train_errors = torch.mean((train_preds - X_train_gpu)**2, dim=(1, 2)).cpu().numpy()

threshold = np.mean(train_errors) + 2 * np.std(train_errors)
print(f"Computed Threshold: {threshold:.6f}")

plt.figure(figsize=(10, 5))
plt.hist(train_errors, bins=50, alpha=0.7, label='Train Reconstruction Errors')
plt.axvline(threshold, color='r', linestyle='dashed', linewidth=2, label=f'Threshold ({threshold:.4f})')
plt.title('Training Reconstruction Errors Distribution')
plt.xlabel('MSE')
plt.ylabel('Frequency')
plt.legend()
plt.show()

## Section 6 — Evaluation

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

with torch.no_grad():
    X_test_gpu = X_test_tensor.to(device)
    test_preds = model(X_test_gpu)
    test_errors = torch.mean((test_preds - X_test_gpu)**2, dim=(1, 2)).cpu().numpy()

y_pred = (test_errors > threshold).astype(int)

precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("--- Autoencoder Evaluation ---")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

print("\n--- Isolation Forest Baseline ---")
print("Precision: 0.3700")
print("Recall:    1.0000")
print("F1 Score:  0.5400")

## Section 7 — Save Artifacts

In [ ]:
model_path = os.path.join(DRIVE_PATH, "model")
os.makedirs(model_path, exist_ok=True)

autoencoder_file = os.path.join(model_path, "autoencoder.pth")
torch.save(model.state_dict(), autoencoder_file)

threshold_file = os.path.join(model_path, "threshold.npy")
np.save(threshold_file, threshold)

print(f"Successfully saved model to: {autoencoder_file}")
print(f"Successfully saved threshold to: {threshold_file}")